# Man vs Machine on Oslo Bors - Data


In [48]:
import pandas as pd
import numpy as np
import pyreadr
import warnings
warnings.filterwarnings('ignore')


### 1. Data cleaning / engineering


In [49]:
# load the data
equity_df = pd.read_csv('../Data/monthly_equity_combined.csv', encoding='latin1')

# rename the first column and drop the last column
equity_df.rename(columns={equity_df.columns[0]: 'TradeDate'}, inplace=True)
equity_df.drop(columns=equity_df.columns[-1], inplace=True)

equity_df


,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,LogReturnAdjGeneric,OffShareTurnover,OffTurnover,NonOffShareTurnover,NonOffTurnover,SharesIssued,DivFactor,CumDivFactor,LastQAccount,LastYAccount
0,1980-01-02 00:00:00,6000,NET,NO0003069908,Nettbuss Sør,1,Ordinary Shares,1,OSE,2214.0,...,NaN,NaN,NaN,NaN,NaN,4000.0,NaN,0.897183,NaN,NaN
1,1980-01-02 00:00:00,6006,AFK,NO0003572802,Arendals Fossekompani,1,Ordinary Shares,1,OSE,1007.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.203478,NaN,NaN
2,1980-01-02 00:00:00,6007,AKE,NO0003514002,Aker RGI A,2,A Shares,1,OSE,1939.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.611575,NaN,NaN
3,1980-01-02 00:00:00,6019,AWS,NO0003083107,Awilco ser. A,2,A Shares,1,OSE,2218.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.382215,NaN,NaN
4,1980-01-02 00:00:00,6026,BEL,NO0003094104,Belships,1,Ordinary Shares,1,OSE,2221.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.586987,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162127,2020-11-27 00:00:00,1305295,BWE,BMG0702P1086,BW Energy Limited,1,Ordinary Shares,1,OSE,12748.0,...,0.392772,5355947.0,1.061510e+08,2654797.0,5.541827e+07,234304300.0,NaN,1.000000,NaN,NaN
162128,2020-11-27 00:00:00,1305313,NOL,BMG6682J1036,Northern Ocean Ltd.,1,Ordinary Shares,1,OSE,12750.0,...,0.566395,9154182.0,6.328005e+07,100844.0,6.092200e+05,63802378.0,NaN,1.000000,NaN,NaN
162129,2020-11-27 00:00:00,1305435,PEXIP,NO0010840507,Pexip Holding,1,Ordinary Shares,1,OSE,12767.0,...,-0.105278,13306870.0,8.377522e+08,1590196.0,1.018904e+08,101563487.0,NaN,1.000000,NaN,NaN
162130,2020-11-27 00:00:00,1305713,LINK,NO0010894231,Link Mobility Group Holding,1,Ordinary Shares,1,OSE,12811.0,...,-0.019803,3439046.0,1.826526e+08,4001558.0,2.146335e+08,270911039.0,NaN,1.000000,NaN,NaN


In [50]:
# standardize ISIN codes
equity_df['ISIN'] = equity_df['ISIN'].str.upper().str.strip()

# align the monthly dates to the end of the month
equity_df['TradeDate'] = pd.to_datetime(equity_df['TradeDate'], errors='coerce') + pd.offsets.MonthEnd(0)

equity_df

,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,LogReturnAdjGeneric,OffShareTurnover,OffTurnover,NonOffShareTurnover,NonOffTurnover,SharesIssued,DivFactor,CumDivFactor,LastQAccount,LastYAccount
0,1980-01-31,6000,NET,NO0003069908,Nettbuss Sør,1,Ordinary Shares,1,OSE,2214.0,...,NaN,NaN,NaN,NaN,NaN,4000.0,NaN,0.897183,NaN,NaN
1,1980-01-31,6006,AFK,NO0003572802,Arendals Fossekompani,1,Ordinary Shares,1,OSE,1007.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.203478,NaN,NaN
2,1980-01-31,6007,AKE,NO0003514002,Aker RGI A,2,A Shares,1,OSE,1939.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.611575,NaN,NaN
3,1980-01-31,6019,AWS,NO0003083107,Awilco ser. A,2,A Shares,1,OSE,2218.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.382215,NaN,NaN
4,1980-01-31,6026,BEL,NO0003094104,Belships,1,Ordinary Shares,1,OSE,2221.0,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.586987,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162127,2020-11-30,1305295,BWE,BMG0702P1086,BW Energy Limited,1,Ordinary Shares,1,OSE,12748.0,...,0.392772,5355947.0,1.061510e+08,2654797.0,5.541827e+07,234304300.0,NaN,1.000000,NaN,NaN
162128,2020-11-30,1305313,NOL,BMG6682J1036,Northern Ocean Ltd.,1,Ordinary Shares,1,OSE,12750.0,...,0.566395,9154182.0,6.328005e+07,100844.0,6.092200e+05,63802378.0,NaN,1.000000,NaN,NaN
162129,2020-11-30,1305435,PEXIP,NO0010840507,Pexip Holding,1,Ordinary Shares,1,OSE,12767.0,...,-0.105278,13306870.0,8.377522e+08,1590196.0,1.018904e+08,101563487.0,NaN,1.000000,NaN,NaN
162130,2020-11-30,1305713,LINK,NO0010894231,Link Mobility Group Holding,1,Ordinary Shares,1,OSE,12811.0,...,-0.019803,3439046.0,1.826526e+08,4001558.0,2.146335e+08,270911039.0,NaN,1.000000,NaN,NaN


In [51]:
# check the security types
print(equity_df['SecurityType'].unique())

# check the distribution of the security types
print(equity_df['SecurityType'].value_counts())

['Ordinary Shares' 'A Shares' 'B Shares' 'New Shares' 'Other'
 'Converted Shares' 'Free Shares' 'Primary Capital Certificates'
 'Warrant - Tegningsrett' 'Converted F Shares' 'Converted B Shares'
 'New B Shares' 'Preference Shares' 'Converted A Shares'
 'Converted Primary Capital Certificates' 'Warrant - European Call'
 'Warrant - European Put' 'Warrant - Index Warrant'
 'Exchange traded funds' 'Warrant - American Call'
 'Warrant - Exchange tradable notes' 'Warrant - Bull ETN'
 'Warrant - Bear ETN']
SecurityType
Ordinary Shares                           78878
Warrant - European Call                   25158
Other                                     19236
Warrant - European Put                     9208
Primary Capital Certificates               6157
Warrant - Bull ETN                         5185
Warrant - Bear ETN                         5174
B Shares                                   4296
A Shares                                   3570
Warrant - Exchange tradable notes          1823
Exc

In [52]:
# define the relevant equity types
keep_types = [
    'Ordinary Shares', 'A Shares', 'B Shares', 'Free Shares', 
    'Primary Capital Certificates', 'Converted Shares', 'Preference Shares'
]

# filter the DataFrame to keep only the relevant equity types
equity_df = equity_df[equity_df['SecurityType'].isin(keep_types)]

print(equity_df['SecurityType'].unique())

['Ordinary Shares' 'A Shares' 'B Shares' 'Converted Shares' 'Free Shares'
 'Primary Capital Certificates' 'Preference Shares']


In [53]:
# find the last date for each ISIN
last_dates = equity_df.groupby('ISIN')['TradeDate'].max().reset_index()

# merge the last dates back to the original DataFrame
equity_df = pd.merge(equity_df, last_dates, on='ISIN', suffixes=('', '_Last'))

# see the distribution of the last dates
equity_df.groupby('ISIN')['TradeDate_Last'].first().value_counts().sort_index()

TradeDate_Last
1985-03-31      1
1985-09-30      1
1986-03-31      2
1986-04-30      2
1986-05-31      3
             ... 
2020-03-31      1
2020-05-31      1
2020-07-31      2
2020-09-30      1
2020-11-30    219
Name: count, Length: 300, dtype: int64

In [54]:
# sort by ISIN and TradeDate
equity_df_sorted = equity_df.sort_values(by=['ISIN', 'TradeDate'])

# calculate the difference in TradeDate for each ISIN
equity_df_sorted['TradeDate_Diff'] = equity_df_sorted.groupby('ISIN')['TradeDate'].diff()


In [55]:
# flag the row where with the last date before the gap as delisted
equity_df_sorted['Delisted'] = False

# flag the last observation before a gap (>32 days) within each ISIN
gap_after = equity_df_sorted.groupby('ISIN')['TradeDate_Diff'].shift(-1) > pd.Timedelta(days=32)
equity_df_sorted.loc[gap_after.fillna(False), 'Delisted'] = True

equity_df_sorted

,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,NonOffShareTurnover,NonOffTurnover,SharesIssued,DivFactor,CumDivFactor,LastQAccount,LastYAccount,TradeDate_Last,TradeDate_Diff,Delisted
27916,1996-07-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,NaN,67292090.0,NaN,1.0,NaN,6242.0,1997-01-31,NaT,False
28085,1996-07-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,489291.0,3.213452e+07,67456832.0,NaN,1.0,NaN,6242.0,1997-01-31,0 days,False
28268,1996-08-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,1175838.0,7.629034e+07,67456832.0,NaN,1.0,NaN,6242.0,1997-01-31,31 days,False
28454,1996-09-30,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,1625380.0,1.037842e+08,67456832.0,NaN,1.0,NaN,6242.0,1997-01-31,30 days,False
28641,1996-10-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,1384771.0,8.428263e+07,67456832.0,NaN,1.0,NaN,6242.0,1997-01-31,31 days,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57389,2007-08-31,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,106.0,4.092000e+03,21500000.0,NaN,1.0,16989.0,17014.0,2007-12-31,31 days,False
57645,2007-09-30,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,NaN,21500000.0,NaN,1.0,16989.0,17014.0,2007-12-31,30 days,False
57908,2007-10-31,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,16793818.0,1.334758e+09,21500000.0,NaN,1.0,17264.0,17014.0,2007-12-31,31 days,False
58173,2007-11-30,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,NaN,21500000.0,NaN,1.0,17264.0,17014.0,2007-12-31,30 days,False


In [56]:
# flag bankrupcy or m&a delisting for last dates of each isin
latest_date = equity_df_sorted['TradeDate'].max()

# only consider the terminal row per ISIN, excluding ISINs that are alive at dataset end
eligible_last_row = (
    equity_df_sorted['TradeDate'].eq(equity_df_sorted['TradeDate_Last']) &
    equity_df_sorted['TradeDate_Last'].ne(latest_date)
)

ma_mask = eligible_last_row & equity_df_sorted['ReturnGeneric'].ge(-0.5)
bankruptcy_mask = eligible_last_row & ~equity_df_sorted['ReturnGeneric'].ge(-0.5)

equity_df_sorted['Delisted_Bankruptcy_MA'] = False
equity_df_sorted['Delist_Type'] = np.nan

equity_df_sorted.loc[ma_mask | bankruptcy_mask, 'Delisted_Bankruptcy_MA'] = True
equity_df_sorted.loc[ma_mask, 'Delist_Type'] = 'M&A'
equity_df_sorted.loc[bankruptcy_mask, 'Delist_Type'] = 'Bankruptcy'

# if bankruptcy and missing ReturnGeneric, impute as requested
equity_df_sorted.loc[bankruptcy_mask & equity_df_sorted['ReturnGeneric'].isna(), 'ReturnGeneric'] = -0.9


In [57]:
# count the number of M&A and bankruptcy delistings
delist_counts = equity_df_sorted['Delist_Type'].value_counts()
print(delist_counts)

# count the number of bankruptcy delistings with imputed ReturnGeneric
imputed_bankruptcy_count = equity_df_sorted[(equity_df_sorted['Delist_Type'] == 'Bankruptcy') & (equity_df_sorted['ReturnGeneric'] == -0.9)].shape[0]
print(f"Number of bankruptcy delistings with imputed ReturnGeneric: {imputed_bankruptcy_count}")

Delist_Type
M&A           634
Bankruptcy     27
Name: count, dtype: int64
Number of bankruptcy delistings with imputed ReturnGeneric: 18


In [58]:
# set spell id for each ISIN
equity_df_sorted['SpellID'] = equity_df_sorted.groupby('ISIN')['TradeDate_Diff'].apply(
    lambda x: (x > pd.Timedelta(days=32)).cumsum() + 1
).values

equity_df_sorted

,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,DivFactor,CumDivFactor,LastQAccount,LastYAccount,TradeDate_Last,TradeDate_Diff,Delisted,Delisted_Bankruptcy_MA,Delist_Type,SpellID
27916,1996-07-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,1.0,NaN,6242.0,1997-01-31,NaT,False,False,NaN,1
28085,1996-07-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,1.0,NaN,6242.0,1997-01-31,0 days,False,False,NaN,1
28268,1996-08-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,1.0,NaN,6242.0,1997-01-31,31 days,False,False,NaN,1
28454,1996-09-30,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,1.0,NaN,6242.0,1997-01-31,30 days,False,False,NaN,1
28641,1996-10-31,16346,RGI,ANN7425Q1095,RGI (Antilles),1,Ordinary Shares,1,OSE,5101.0,...,NaN,1.0,NaN,6242.0,1997-01-31,31 days,False,False,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57389,2007-08-31,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,1.0,16989.0,17014.0,2007-12-31,31 days,False,False,NaN,1
57645,2007-09-30,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,1.0,16989.0,17014.0,2007-12-31,30 days,False,False,NaN,1
57908,2007-10-31,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,1.0,17264.0,17014.0,2007-12-31,31 days,False,False,NaN,1
58173,2007-11-30,64808,FRID,VGG3724W1014,Frigstad Discoverer Invest,1,Ordinary Shares,1,OSE,8275.0,...,NaN,1.0,17264.0,17014.0,2007-12-31,30 days,False,False,NaN,1


In [59]:
# save the final DataFrame to a new CSV file
equity_df_sorted.to_csv('../Data/processed_equity_data.csv', index=False)

### 2. Feature engineering

In [60]:
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# Assuming your dataframe is named 'df'
equity_df_sorted['TradeDate'] = pd.to_datetime(equity_df_sorted['TradeDate'])

# Ensure strict sorting before calculating rolling features
equity_df_sorted.sort_values(by=['ISIN', 'SpellID', 'TradeDate'], inplace=True)

# -------------------------------------------------------------------
# 1. WINSORIZE TARGET (Cap at 1st and 99th Percentile)
# -------------------------------------------------------------------
# This neutralizes the +59,300% anomalies.
lower_bound = equity_df_sorted['ReturnAdjGeneric'].quantile(0.01)
upper_bound = equity_df_sorted['ReturnAdjGeneric'].quantile(0.99)
equity_df_sorted['ReturnAdjGeneric_win'] = equity_df_sorted['ReturnAdjGeneric'].clip(lower=lower_bound, upper=upper_bound)

# Create the Target by shifting the already-winsorized return
equity_df_sorted['Target'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].shift(-1)

# Create a non-winsorized target for comparison (optional)
equity_df_sorted['Target_raw'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric'].shift(-1)

# -------------------------------------------------------------------
# 2. SIZE & LIQUIDITY FEATURES
# -------------------------------------------------------------------
# Market Cap = Price (Generic) * SharesIssued
equity_df_sorted['mkt_cap'] = equity_df_sorted['Generic'] * equity_df_sorted['SharesIssued']
# Log Size normalizes the exponential difference between micro-caps and Equinor
equity_df_sorted['log_size'] = np.log(equity_df_sorted['mkt_cap'].replace(0, np.nan))

# Turnover = Shares Traded / Total Shares Issued (proxy for liquidity)
# Using 'OffShareTurnover' or 'NonOffShareTurnover' depending on your preference
equity_df_sorted['turnover'] = equity_df_sorted['OffShareTurnover'] / equity_df_sorted['SharesIssued']

# -------------------------------------------------------------------
# 3. MOMENTUM & REVERSAL (Spell-Aware)
# -------------------------------------------------------------------
# Short-term Reversal: The return of the most recent month (t-1)
equity_df_sorted['rev_1m'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].shift(1)

# Intermediate Momentum (6m and 12m):
# Standard finance literature skips the most recent month (t-1) to isolate momentum from reversal.
# We shift(2) to start from t-2, then sum the previous 5 or 11 months.
equity_df_sorted['mom_6m'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].transform(
    lambda x: x.shift(2).rolling(5).sum()
)

equity_df_sorted['mom_12m'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].transform(
    lambda x: x.shift(2).rolling(11).sum()
)

# -------------------------------------------------------------------
# 4. RISK METRICS (Volatility)
# -------------------------------------------------------------------
# 3-Month and 6-Month Realized Volatility
equity_df_sorted['vol_3m'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].transform(
    lambda x: x.shift(1).rolling(3).std()
)
equity_df_sorted['vol_6m'] = equity_df_sorted.groupby(['ISIN', 'SpellID'])['ReturnAdjGeneric_win'].transform(
    lambda x: x.shift(1).rolling(6).std()
)


print("Factor Zoo successfully created!")

Factor Zoo successfully created!


In [61]:
# 1. Calculate the absolute NOK value traded that month
# Generic (Price) * OffShareTurnover (Shares Traded)
equity_df_sorted['NOK_volume'] = equity_df_sorted['Generic'] * equity_df_sorted['OffShareTurnover']

# 2. Cross-sectionally rank the NOK volume each month
equity_df_sorted['liquidity_rank'] = equity_df_sorted.groupby('TradeDate')['NOK_volume'].rank(pct=True)

# 3. Drop the most illiquid/un-tradable bottom 20% of the market
investable_universe = equity_df_sorted[equity_df_sorted['liquidity_rank'] >= 0.10].copy()

In [62]:
# 1. Define the list of raw features you just created
features_to_rank = [
    'log_size', 'turnover', 
    'rev_1m', 'mom_6m', 'mom_12m', 
    'vol_3m', 'vol_6m'
]

# 2. Loop through each feature and create a new ranked column
for col in features_to_rank:
    # rank(pct=True) gives a value from 0.0 to 1.0 within each month
    investable_universe[f'{col}_rank'] = investable_universe.groupby('TradeDate')[col].rank(pct=True)

# 3. Drop the rows that have NaNs in the ranked features
# This automatically removes the first 5-11 months of new spells where 
# rolling features like mom_12m couldn't be calculated.
ranked_cols = [f'{col}_rank' for col in features_to_rank]
investable_universe = investable_universe.dropna(subset=ranked_cols)

# Optional: You can drop the raw feature columns now to save memory, 
# as the ML model will ONLY use the ranked versions.
# investable_universe = investable_universe.drop(columns=features_to_rank)

In [ ]:
# -------------------------------------------------------------------
# 5. CROSS-SECTIONAL Z-SCORING - *ADDED*
# -------------------------------------------------------------------
features_to_zscore = ['log_size', 'turnover', 'rev_1m', 'mom_6m', 'mom_12m', 'vol_3m', 'vol_6m', 'Target']

def cs_zscore(df, col):
    # (Value - Mean) / Standard Deviation, grouped by TradeDate
    return df.groupby('TradeDate')[col].transform(lambda x: (x - x.mean()) / x.std())

for col in features_to_zscore:
    investable_universe[f'{col}_z'] = cs_zscore(investable_universe, col)


# -------------------------------------------------------------------
# 6. DROP NaNs (End of pipeline)
# -------------------------------------------------------------------
# Drop the missing targets at the end of the spells
investable_universe = investable_universe.dropna(subset=['Target', 'Target_raw'])

# Drop rows missing the longest rolling feature (first 11 months of new spells)
investable_universe = investable_universe.dropna(subset=['mom_12m_z'])

print("Factor Zoo perfectly created and ready for ML!")

Factor Zoo perfectly created and ready for ML!


In [64]:
# save the final DataFrame to a new CSV file
investable_universe.to_csv('../Data/investable_universe.csv', index=False)

In [65]:
investable_universe

,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,vol_3m_rank,vol_6m_rank,log_size_z,turnover_z,rev_1m_z,mom_6m_z,mom_12m_z,vol_3m_z,vol_6m_z,Target_z
79810,2015-06-30,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,0.289474,0.989247,-0.853921,1.241796,-0.044167,-0.374240,-1.403878,-0.668727,2.769008,-0.591769
80027,2015-07-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,0.343915,0.983784,-0.865093,0.291696,-0.470742,0.473789,-1.688070,-0.533116,2.592352,-0.034180
80244,2015-08-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,0.378947,0.697297,-0.825407,0.385545,-0.580609,-1.542791,-1.388849,-0.519248,0.352740,-0.575475
80459,2015-09-30,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,0.046875,0.162162,-0.864670,0.983566,-0.012080,-0.614041,-0.984236,-1.041243,-0.857283,-2.289543
80674,2015-10-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,0.046392,0.196809,-1.050228,3.749652,-0.625084,-0.547208,-1.202919,-1.028363,-0.799767,-0.594089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36549,1999-08-31,30500,TTI,USU872831040,Tecmar Technologies Int.,1,Ordinary Shares,1,OSE,6109.0,...,0.844221,0.613065,-1.760785,0.183414,1.495818,-1.957443,-2.492174,1.025967,0.072922,0.010846
36784,1999-09-30,30500,TTI,USU872831040,Tecmar Technologies Int.,1,Ordinary Shares,1,OSE,6109.0,...,0.980296,0.846535,-1.736302,0.456384,-3.318609,-1.587887,-2.025023,2.600513,0.940085,-1.490438
37019,1999-10-31,30500,TTI,USU872831040,Tecmar Technologies Int.,1,Ordinary Shares,1,OSE,6109.0,...,0.979695,0.865285,-1.882837,0.270060,-0.012624,-2.441935,-2.235439,2.619772,1.074382,2.939117
37249,1999-11-30,30500,TTI,USU872831040,Tecmar Technologies Int.,1,Ordinary Shares,1,OSE,6109.0,...,0.918367,0.921875,-1.412876,4.143830,-1.522230,-2.252912,-1.670708,1.442422,1.433039,-2.871146


### 3. Merging "Best Ideas" data

In [66]:
import pandas as pd

# -------------------------------------------------------------------
# STEP 1: LOAD AND CLEAN THE MUTUAL FUND DATA
# -------------------------------------------------------------------
portfolios_df = pd.read_csv('../Data/portfolios_cleaned.csv')

# Drop any rows where the stock ISIN is missing (e.g., Cash or external bonds)
portfolios_df = portfolios_df.dropna(subset=['stock_isin']).copy()

# Align the portfolio dates to the End-of-Month to match your Master Equity Panel
portfolios_df['date'] = pd.to_datetime(portfolios_df['date']) + pd.offsets.MonthEnd(0)

# -------------------------------------------------------------------
# STEP 2: CALCULATE "BEST IDEAS" (TOP 10 CONVICTION)
# -------------------------------------------------------------------
# Rank the holdings within every single fund, every single month based on weight
portfolios_df['Fund_Rank'] = portfolios_df.groupby(['fund_name', 'date'])['weight'].rank(ascending=False, method='min')

# Create a binary flag: 1 if it is a Top 10 holding for that fund, 0 otherwise
portfolios_df['Is_Top_10'] = (portfolios_df['Fund_Rank'] <= 10).astype(int)

# -------------------------------------------------------------------
# STEP 3: AGGREGATE TO THE STOCK-MONTH LEVEL
# -------------------------------------------------------------------
# We group by ISIN and Date to see the overall "Smart Money" sentiment for each stock
bi_features = portfolios_df.groupby(['stock_isin', 'date']).agg(
    BI_Avg_Weight=('weight', 'mean'),        # The average allocation size
    BI_Fund_Count=('fund_name', 'nunique'),  # Breadth: How many funds own it?
    BI_Top10_Count=('Is_Top_10', 'sum')      # Conviction: How many funds have it in Top 10?
).reset_index()

# -------------------------------------------------------------------
# STEP 4: MERGE WITH YOUR POST-2010 INVESTABLE UNIVERSE
# -------------------------------------------------------------------
# Create the Modern Track (2010 onwards) from your clean equity dataframe
# (Assuming your clean df from the last step is named `equity_df_sorted`)
modern_equity_df = investable_universe[investable_universe['TradeDate'] >= '2010-01-31'].copy()

# Left merge the Best Ideas data onto the stock panel
modern_equity_df = pd.merge(
    modern_equity_df, 
    bi_features, 
    left_on=['ISIN', 'TradeDate'], 
    right_on=['stock_isin', 'date'], 
    how='left'
)

# -------------------------------------------------------------------
# STEP 5: FILL NaNs AND Z-SCORE THE SMART MONEY FEATURES
# -------------------------------------------------------------------
# If a stock is not held by ANY mutual fund, it gets a NaN from the merge.
# We must fill these with 0, because 0 funds = 0 weight, 0 count, 0 top 10s.
smart_money_cols = ['BI_Avg_Weight', 'BI_Fund_Count', 'BI_Top10_Count']
for col in smart_money_cols:
    modern_equity_df[col] = modern_equity_df[col].fillna(0)

# Finally, Cross-Sectionally Z-Score these new features just like Momentum!
# This ensures XGBoost compares "Fund Conviction" directly against "Momentum" on the exact same scale.
for col in smart_money_cols:
    modern_equity_df[f'{col}_z'] = modern_equity_df.groupby('TradeDate')[col].transform(lambda x: (x - x.mean()) / x.std())

# Clean up redundant columns from the merge
modern_equity_df = modern_equity_df.drop(columns=['stock_isin', 'date'])

print(f"Modern Dataset successfully created with shape: {modern_equity_df.shape}")
print("Smart Money factors engineered and Z-scored!")

Modern Dataset successfully created with shape: (23970, 83)
Smart Money factors engineered and Z-scored!


In [70]:
# save the final DataFrame to a new CSV file
modern_equity_df.to_csv('../Data/modern_equity_dataset.csv', index=False)

In [68]:
modern_equity_df

,TradeDate,SecurityId,Symbol,ISIN,SecurityName,SecurityTypeId,SecurityType,IsStock,Market,CompanyId,...,mom_12m_z,vol_3m_z,vol_6m_z,Target_z,BI_Avg_Weight,BI_Fund_Count,BI_Top10_Count,BI_Avg_Weight_z,BI_Fund_Count_z,BI_Top10_Count_z
0,2015-06-30,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,-1.403878,-0.668727,2.769008,-0.591769,0.0,0.0,0.0,-0.814961,-0.776399,-0.378889
1,2015-07-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,-1.688070,-0.533116,2.592352,-0.034180,0.0,0.0,0.0,-0.801001,-0.761450,-0.363026
2,2015-08-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,-1.388849,-0.519248,0.352740,-0.575475,0.0,0.0,0.0,-0.805786,-0.759229,-0.367677
3,2015-09-30,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,-0.984236,-1.041243,-0.857283,-2.289543,0.0,0.0,0.0,-0.826491,-0.762615,-0.374404
4,2015-10-31,1301079,PNOR,AU0000057408,PetroNor E&P Limited,1,Ordinary Shares,1,OAX,12300.0,...,-1.202919,-1.028363,-0.799767,-0.594089,0.0,0.0,0.0,-0.835190,-0.765700,-0.369304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23965,2020-06-30,58337,GIG,US36467X2062,Gaming Innovation Group,1,Ordinary Shares,1,OSE,7829.0,...,-1.097637,1.002171,1.597786,1.729360,0.0,0.0,0.0,-0.812730,-0.713789,-0.378234
23966,2020-07-31,58337,GIG,US36467X2062,Gaming Innovation Group,1,Ordinary Shares,1,OSE,7829.0,...,-0.476730,2.440788,1.520248,0.009327,0.0,0.0,0.0,-0.822262,-0.706535,-0.385086
23967,2020-08-31,58337,GIG,US36467X2062,Gaming Innovation Group,1,Ordinary Shares,1,OSE,7829.0,...,-0.705040,2.588473,1.513668,0.381859,0.0,0.0,0.0,-0.814593,-0.696486,-0.376404
23968,2020-09-30,58337,GIG,US36467X2062,Gaming Innovation Group,1,Ordinary Shares,1,OSE,7829.0,...,0.265488,1.382707,1.138733,0.125022,0.0,0.0,0.0,-0.818864,-0.698746,-0.385225
